# k03 — Memory across sessions

Remember facts in one session; recall them in a completely fresh one.
The store is the bridge: `<data_dir>/memory.sqlite3` outlives every
agent, session, and process that touches it (the `finstack-know` CLI
writes the same file).

Trust: T2 callback. Network: none (scripted models).

In [ ]:
import tempfile
from pathlib import Path

import finstack_ai
from _knowledge import build_knowledge_agent, knowledge_memory, scripted_model

workdir = Path(tempfile.mkdtemp(prefix="finstack-know-k03-"))

## Session one: store two facts

A scripted model drives the `remember` tool twice, then finishes. Scope
is bound to the extension — the model chooses bodies and keywords, never
where memories land.

In [ ]:
writer_script = [
    {
        "text": "",
        "tool_calls": [
            {
                "name": "remember",
                "arguments": {
                    "id": "close-deadline",
                    "keywords": ["quarterly", "close", "deadline"],
                    "body": "The quarterly close deadline is the fifth business day.",
                },
            }
        ],
    },
    {
        "text": "",
        "tool_calls": [
            {
                "name": "remember",
                "arguments": {
                    "id": "audit-partner",
                    "keywords": ["audit", "partner"],
                    "body": "The audit partner is Meridian LLP.",
                },
            }
        ],
    },
    "Both facts stored.",
]

writer = (await build_knowledge_agent(workdir, scripted_model(writer_script))).agent
stored = await writer.run("Remember the close deadline and the audit partner.")
print(stored.text)
assert "stored" in stored.text.lower()

## A fresh session recalls

A brand-new agent over the same data directory. Before its model runs,
the recall provider searches the store with the user input and
contributes matching records as bounded `[memory]` context items — the
capture below shows exactly what the model saw.

In [ ]:
seen: list[dict] = []


async def _capture(context, request):
    del context
    seen.append(request)
    return {
        "text": "The close deadline is the fifth business day.",
        "completion_id": "k03-recall",
    }


reader = (
    await build_knowledge_agent(
        workdir,
        finstack_ai.PythonModel(
            _capture,
            component="knowledge.model.k03-reader",
            provider="knowledge-scripted",
            model="knowledge-scripted-model",
            context_window_tokens=131_072,
        ),
    )
).agent
answer = await reader.run("when is the quarterly close deadline?")
print(answer.text)

recalled = [
    block["text"]
    for message in seen[0]["messages"]
    for block in message.get("content", [])
    if '"source":"memory"' in block.get("text", "")
]
print(recalled)
assert any("fifth business day" in text for text in recalled)

## The recall budget

Recall is bounded, not a dump of the store. `max_hits` caps how many
records one turn may contribute; relevance ranking decides which. With
`max_hits=1`, a query matching both facts still contributes exactly
one.

In [ ]:
memory = knowledge_memory(workdir)
seen_budget: list[dict] = []


async def _capture_budget(context, request):
    del context
    seen_budget.append(request)
    return {"text": "one memory only", "completion_id": "k03-budget"}


tight = await finstack_ai.Agent.from_python(
    finstack_ai.PythonModel(
        _capture_budget,
        component="knowledge.model.k03-budget",
        provider="knowledge-scripted",
        model="knowledge-scripted-model",
        context_window_tokens=131_072,
    ),
    context_providers=[memory.context_provider(max_hits=1)],
)
await tight.run("quarterly close deadline and audit partner")

budget_recalled = [
    block["text"]
    for message in seen_budget[0]["messages"]
    for block in message.get("content", [])
    if block.get("text", "").endswith("[memory]")
]
print(budget_recalled)
assert len(budget_recalled) == 1

In [ ]:
import shutil

shutil.rmtree(workdir, ignore_errors=True)
print("cleaned", workdir)